# Model Comparison: Blog Search Agent Event Coverage & Cost

Compare event detection quality and cost across model configurations:
- **Baseline**: Claude Opus 4.6 across all swap points (assumed to detect all actual events)
- **Challenger 1**: GLM 5 across all swap points
- **Challenger 2**: Haiku 4.5 across all swap points
- **Challenger 3**: Opus 4.6 (agent) + Haiku 4.5 (RSS) + Haiku 4.5 (web) across all swap points
- **Current Default**: Sonnet 4.6 (agent) + Nova 2 Lite (RSS) + Haiku 4.5 (web)

Kimi K2.5 blocked by SCP. Qwen VL 235B was not able to call the web search tool, seemed to be a model capability issue.

Nova 2 Lite as the orchestrator hit max token limits

In [ ]:
!uv pip install requests boto3 feedparser pydantic python-dotenv strands-agents mcp openpyxl pandas matplotlib


In [ ]:
import sys
from pathlib import Path

EVAL_DIR = Path.cwd() if Path.cwd().name == "evaluation" else Path.cwd() / "evaluation"
ROOT = EVAL_DIR.parent

sys.path.insert(0, str(EVAL_DIR))
sys.path.insert(0, str(ROOT / "blog_search"))

from scripts.registry_sampler import sample_teams
from scripts.runner import run_evaluation, RESULTS_DIR
from scripts.comparator import compare_runs
from scripts.pricing import MODEL_CONFIGS, compute_cost_breakdown

## 1. Configuration

In [ ]:
n_teams=20
LOOKBACK_HOURS = 168
REFERENCE_DATE = ""  # Leave empty for current time, or set e.g. "2026-08-04"

# ---
# teams_data = sample_teams(n=n_teams, seed=42, min_non_rss=2)
# team_names = [t["team"] for t in teams_data]
# print(team_names)



In [ ]:
team_names = ['Eastern Michigan Eagles', 'Auburn Tigers', 'Missouri Tigers', 'Maine Black Bears', 'La Salle Explorers', 'George Washington Revolutionaries', 'Duke Blue Devils', 'Columbia Lions', 'Texas Longhorns', 'BYU Cougars', 'Austin Peay Governors', 'Dartmouth Big Green', 'Kentucky Wildcats', 'Long Island University Sharks', 'West Georgia Wolves', 'Iowa State Cyclones', 'Texas A&M Aggies', 'LSU Tigers', 'UAB Blazers', "Mount St. Mary's Mountaineers"]

## 2. Run Baseline — Claude Opus 4.6

In [ ]:
baseline_result = await run_evaluation(
    teams=team_names,
    model_config=MODEL_CONFIGS["baseline_opus"],
    lookback_hours=LOOKBACK_HOURS,
    reference_date=REFERENCE_DATE,
)
print(f"\nBaseline: {baseline_result['totals']['events']} events, ${baseline_result['totals']['cost_usd']:.4f}")

## 3. Run Challenger 1 — GLM 5

In [ ]:
import requests, os
from dotenv import load_dotenv
load_dotenv("../blog_search/.env")

domain = os.environ["COGNITO_DOMAIN"]
client_id = os.environ["COGNITO_CLIENT_ID"]  
client_secret = os.environ["COGNITO_CLIENT_SECRET"]

resp = requests.post(
    f"https://{domain}/oauth2/token",
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    auth=(client_id, client_secret),
    data={"grant_type": "client_credentials"},
    timeout=10,
)
print(resp.status_code, resp.json().get("access_token", "NO TOKEN")[:50])

In [ ]:
# import sys
# sys.path.insert(0, "../blog_search")
# import blog_search_agent
# blog_search_agent._gateway_client = None
# blog_search_agent._gateway_token = None

import time
# time.sleep(60)
glm_result = await run_evaluation(
    teams=team_names,
    model_config=MODEL_CONFIGS["challenger_glm"],
    lookback_hours=LOOKBACK_HOURS,
    reference_date=REFERENCE_DATE,
)
print(f"\nGLM 5: {glm_result['totals']['events']} events, ${glm_result['totals']['cost_usd']:.4f}")

## 4. Run Challenger 2 — Haiku 4.5

In [ ]:
haiku_result = await run_evaluation(
    teams=team_names,
    model_config=MODEL_CONFIGS["challenger_haiku"],
    lookback_hours=LOOKBACK_HOURS,
    reference_date=REFERENCE_DATE,
)
print(f"\nHaiku 4.5: {haiku_result['totals']['events']} events, ${haiku_result['totals']['cost_usd']:.4f}")

## 5. Run Challenger 3 — Opus 4.6 + Haiku 4.5

In [ ]:
opus_haiku_result = await run_evaluation(
    teams=team_names,
    model_config=MODEL_CONFIGS["challenger_opus_haiku"],
    lookback_hours=LOOKBACK_HOURS,
    reference_date=REFERENCE_DATE,
)
print(f"\nNova 2 Lite: {opus_haiku_result['totals']['events']} events, ${opus_haiku_result['totals']['cost_usd']:.4f}")

## 6. Run Current Default — Sonnet 4.6 + Nova Lite + Haiku 4.5

In [ ]:
default_result = await run_evaluation(
    teams=team_names,
    model_config=MODEL_CONFIGS["current_default"],
    lookback_hours=LOOKBACK_HOURS,
    reference_date=REFERENCE_DATE,
)
print(f"\nDefault: {default_result['totals']['events']} events, ${default_result['totals']['cost_usd']:.4f}")

## 7. Coverage Comparison

In [ ]:
import json
from pathlib import Path
opus_result_path = "/Users/priyalex/Documents/projects/fanduel/ncaa-blog-agent/evaluation/results/baseline_opus_20260806_194914.json"
haiku_result_path = "/Users/priyalex/Documents/projects/fanduel/ncaa-blog-agent/evaluation/results/challenger_haiku_20260806_201126.json"
opus_haiku_result_path = "/Users/priyalex/Documents/projects/fanduel/ncaa-blog-agent/evaluation/results/challenger_opus_haiku_20260806_201955.json"
glm_result_path = "/Users/priyalex/Documents/projects/fanduel/ncaa-blog-agent/evaluation/results/challenger_glm_20260806_200153.json"
default_result_path = "/Users/priyalex/Documents/projects/fanduel/ncaa-blog-agent/evaluation/results/current_default_20260806_210952.json"

def load_result(path):
  with open(path) as f:
    return json.load(f)
  
baseline_result = load_result(opus_result_path)
haiku_result = load_result(haiku_result_path)
opus_haiku_result = load_result(opus_haiku_result_path)
glm_result = load_result(glm_result_path)
default_result = load_result(default_result_path)


In [ ]:
# ncaa-blog-agent/evaluation/results/challenger_haiku_20260806_201126.json
# ncaa-blog-agent/evaluation/results/challenger_opus_haiku_20260806_201955.json
# ncaa-blog-agent/evaluation/results/baseline_opus_20260806_194914.json
#  ncaa-blog-agent/evaluation/results/challenger_glm_20260806_200153.json
# ncaa-blog-agent/evaluation/results/current_default_20260806_210952.json

import pandas as pd

glm_comparison = compare_runs(baseline_result, glm_result)
haiku_comparison = compare_runs(baseline_result, haiku_result)
opus_haiku_comparison = compare_runs(baseline_result, opus_haiku_result)
default_comparison = compare_runs(baseline_result, default_result)

all_runs = [
    ("Claude Opus 4.6 (baseline)", baseline_result, None),
    ("GLM 5", glm_result, glm_comparison),
    ("Haiku 4.5", haiku_result, haiku_comparison),
    ("Opus (blog agent) + Haiku (RSS-llm-extraction) + Haiku (Web-llm-extraction)", opus_haiku_result, opus_haiku_comparison),
    ("Sonnet (blog agent) + Nova (RSS-llm-extraction) + Haiku (Web-llm-extraction", default_result, default_comparison),
]

n_teams = len(team_names)

summary_data = []
for label, result, comparison in all_runs:
    recall = comparison["aggregate"]["recall_vs_baseline"] if comparison else 1.0
    summary_data.append({
        "Model": label,
        "Events Found": result["totals"]["events"],
        "Recall vs Baseline": f"{recall:.1%}",
        "Total Web Searches": result["totals"]["web_searches"],
        "Avg Web Searches/Team": f"{result['totals']['web_searches'] / n_teams:.1f}",
        "Cost (USD)": f"${result['totals']['cost_usd']:.4f}",
    })

df_coverage_vs_cost = pd.DataFrame(summary_data)
df_coverage_vs_cost

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ["Opus 4.6\n(baseline)", "GLM 5", "Haiku 4.5", "Opus (blog agent)\n+Haiku (RSS-llm-extraction)\n+Haiku (Web-llm-extraction)", "Sonnet (blog agent)\n+Nova (RSS-llm-extraction)\n+Haiku (Web-llm-extraction)"]
results_list = [baseline_result, glm_result, haiku_result, opus_haiku_result, default_result]
comparisons = [None, glm_comparison, haiku_comparison, opus_haiku_comparison, default_comparison]
colors = ["#4A90D9", "#E67E22", "#27AE60", "#8E44AD", "#E74C3C"]

# Compute cost breakdown per run
breakdowns = []
for result in results_list:
    token_keys = ["agent_input_tokens", "agent_output_tokens",
                  "rss_input_tokens", "rss_output_tokens",
                  "web_input_tokens", "web_output_tokens"]
    totals = {k: 0 for k in token_keys}
    for r in result["team_results"]:
        for k in token_keys:
            totals[k] += r["token_usage"][k]
    bd = compute_cost_breakdown(
        result["model_config"], totals, result["totals"]["web_searches"]
    )
    breakdowns.append(bd)

## 8. Cost Breakdown: Web Search API vs LLM Extraction

Costs are split into:
- **Web Search API**: $7 per 1,000 queries (gateway__WebSearch tool calls)
- **Agent LLM**: Strands orchestrator model (reasoning + tool selection)
- **RSS Extraction LLM**: Model that extracts events from RSS feed content
- **Web Extraction LLM**: Model that extracts events from web-fetched page content

In [ ]:
# Cost breakdown table per team
cost_data = []
for label, result, bd in zip(
    [r[0] for r in all_runs], results_list, breakdowns
):
    total = bd["total"]
    web_searches = result["totals"]["web_searches"]
    cost_data.append({
        "Model": label,
        "Web Search API": f"${bd['web_search']:.4f}",
        "Agent LLM": f"${bd['agent_llm']:.4f}",
        "RSS Extraction LLM": f"${bd['rss_llm']:.4f}",
        "Web Extraction LLM": f"${bd['web_llm']:.4f}",
        "Total": f"${total:.4f}",
        "Cost/Team": f"${total / n_teams:.4f}",
    })

df_cost_breakdown = pd.DataFrame(cost_data)
df_cost_breakdown

In [ ]:
# Stacked bar: 4-component cost breakdown
fig, ax = plt.subplots(figsize=(12, 5))

web_search_costs = [bd["web_search"] for bd in breakdowns]
agent_costs = [bd["agent_llm"] for bd in breakdowns]
rss_costs = [bd["rss_llm"] for bd in breakdowns]
web_extract_costs = [bd["web_llm"] for bd in breakdowns]

x = np.arange(len(models))
bottom = np.zeros(len(models))

ax.bar(x, web_search_costs, label="Web Search API ($7/1K)", color="#E74C3C", bottom=bottom)
bottom += web_search_costs
ax.bar(x, agent_costs, label="Agent LLM (orchestrator)", color="#4A90D9", bottom=bottom)
bottom += agent_costs
ax.bar(x, rss_costs, label="RSS Extraction LLM", color="#E67E22", bottom=bottom)
bottom += rss_costs
ax.bar(x, web_extract_costs, label="Web Extraction LLM", color="#27AE60", bottom=bottom)

ax.set_ylabel("Cost (USD)")
ax.set_title("Cost Breakdown by Component")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend(loc="upper right")

totals = [bd["total"] for bd in breakdowns]
for i, t in enumerate(totals):
    ax.text(i, t + max(totals) * 0.02, f"${t:.4f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()
plt.savefig("results/cost_breakdown.png")

# ## 9. Average Cost per Team & Event Coverage

In [ ]:
# fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# # Avg Cost per Team
# avg_costs = [bd["total"] / n_teams for bd in breakdowns]
# axes[0].bar(models, avg_costs, color=colors)
# axes[0].set_ylabel("Cost per Team (USD)")
# axes[0].set_title("Average Cost per Team")
# for i, v in enumerate(avg_costs):
#     axes[0].text(i, v + max(avg_costs) * 0.02, f"${v:.4f}", ha="center", fontweight="bold")

# # Recall vs Baseline
# recalls = [c["aggregate"]["recall_vs_baseline"] if c else 1.0 for c in comparisons]
# axes[1].bar(models, [r * 100 for r in recalls], color=colors)
# axes[1].set_title("Event Coverage vs Baseline")
# axes[1].set_ylabel("Recall (%)")
# axes[1].set_ylim(0, 110)
# axes[1].axhline(y=100, color="gray", linestyle="--", alpha=0.5)
# for i, v in enumerate(recalls):
#     axes[1].text(i, v * 100 + 2, f"{v:.0%}", ha="center", fontweight="bold")

# plt.tight_layout()
# plt.savefig(str(RESULTS_DIR / "model_comparison_charts.png"), dpi=150, bbox_inches="tight")
# plt.show()

## 12. Cost Efficiency Summary

In [ ]:
print(f"Lookback hours: {LOOKBACK_HOURS}")
print(f"Reference date: {REFERENCE_DATE}")
print(f"no. of teams investigated: {n_teams}")

baseline_cost = baseline_result["totals"]["cost_usd"]

efficiency_data = []
for (label, result, comparison), bd in zip(all_runs, breakdowns):
    cost = bd["total"]
    recall = comparison["aggregate"]["recall_vs_baseline"] if comparison else 1.0
    events = result["totals"]["events"]
    web_searches = result["totals"]["web_searches"]
    web_fetches = sum(r["web_fetches"] for r in result["team_results"])
    savings = ((baseline_cost - cost) / baseline_cost * 100) if baseline_cost > 0 else 0

    efficiency_data.append({
        "N_teams": n_teams,
        "Lookback_hours": LOOKBACK_HOURS,
        "Model": label,
        "Events": events,
        "Recall": f"{recall:.1%}",
        "Cost/Team": f"${cost / n_teams:.4f}",
        "projected_cost_over_all_teams": f"${cost * 365/ n_teams:.0f}",
        "Web Searches": web_searches,
        "Web Fetches": web_fetches,
        "% Cost from Web Search API": f"{bd['web_search'] / cost * 100:.0f}%" if cost > 0 else "—",
        "% Cost from Agent LLM": f"{bd['agent_llm'] / cost * 100:.0f}%" if cost > 0 else "—",
        "% Cost from RSS Extract": f"{bd['rss_llm'] / cost * 100:.0f}%" if cost > 0 else "—",
        "% Cost from Web Extract": f"{bd['web_llm'] / cost * 100:.0f}%" if cost > 0 else "—",
        "Savings vs Baseline": f"{savings:.1f}%",
    })

df_summary = pd.DataFrame(efficiency_data)
f_path = "results/blog_agent_coverage_vs_cost.xlsx"
df_summary.to_excel(f_path, index=False)

In [ ]:
df_summary

# Key observations
1. The blog agent inference & web searches are the biggest cost drivers. 
2. Cost vs coverage tradeoff: the model choice for blog agent also has the most impact on the recall, where the recall shows what fraction of events detected in a given model-combination overlaps with the baseline events (taking the baseline as the ground truth).

2. The `projected_cost_over_all_teams` may differ from what Datadog reports for one run across all 365 teams. Key factors contributing to the difference:
    
    a. The cohort of 20 teams chosen may not be representative of the full team cohort. These 20 teams have at least 2 non-RSS active urls. There can be teams with > 2 non-RSS active urls which will be processed via the agent driving the cost up via its high inference tokens. RSS-feed processing does not lead to any significant costs as it is a non-agentic processing approach.
    
    b. The full run across 365 teams in production environment may have more retries that cost tokens. This analysis with 20 teams did not have any retries.
